# TrendShelf Scoring Leakage Audit

**Purpose:** Check whether the ML model is learning the rule-based opportunity formula instead of an independent signal.

> This is not business outcome validation. The target is rule-derived.

## Section 0 — Setup

In [ ]:
import os, pathlib
os.chdir(r'C:\Users\Hp\Desktop\trendshelf')
os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = str(pathlib.Path('credentials.json').resolve())

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from google.cloud import bigquery
from sklearn.model_selection import StratifiedKFold, KFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (roc_auc_score, precision_score, recall_score,
                              f1_score, accuracy_score, confusion_matrix)
from sklearn.inspection import permutation_importance

client = bigquery.Client(project='windy-container-451804-n4')
os.makedirs('outputs', exist_ok=True)

RANDOM_STATE     = 42
TARGET_THRESHOLD = 65
print('Setup complete.')


## Section 1 — Load Data

In [ ]:
query = '''
SELECT
  aq.store_id,
  aq.category_name,
  aq.overall_opportunity_score,
  aq.opportunity_tier,
  aq.overall_demand_gap_score,
  er.expansion_readiness_score,
  aq.overall_confidence_score        AS confidence_score,
  aq.overall_risk_score,
  aq.pricing_power_score,
  aq.markdown_safety_score,
  pi.price_gap_pct,
  pi.adjusted_price_gap_pct,
  pi.competitor_product_count,
  pi.kroger_product_count,
  pi.price_gap_confidence_weight,
  dt.google_trends_level_score,
  dt.google_trends_momentum_score,
  dt.demand_velocity_score,
  mf.ppi_3mo_trend,
  mf.cpi_3mo_trend
FROM `windy-container-451804-n4.bronze.mart_action_queue` aq
LEFT JOIN `windy-container-451804-n4.bronze.mart_expansion_readiness` er
  ON aq.store_id = er.store_id AND aq.category_name = er.category_name
LEFT JOIN `windy-container-451804-n4.bronze.mart_pricing_intelligence` pi
  ON aq.store_id = pi.store_id AND aq.category_name = pi.category_name
LEFT JOIN `windy-container-451804-n4.bronze.int_demand_trend_features` dt
  ON aq.category_name = dt.category
LEFT JOIN (
  SELECT ppi_3mo_trend, cpi_3mo_trend
  FROM `windy-container-451804-n4.bronze.int_macro_trend_features`
  ORDER BY reference_month DESC LIMIT 1
) mf ON 1=1
'''

df = client.query(query).to_dataframe()
print(f'Shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
print()
nulls = df.isnull().sum()
nulls = nulls[nulls > 0]
print('Missing values:'); print(nulls.to_string() if len(nulls) else '  none')

df['target'] = (df['overall_opportunity_score'] >= TARGET_THRESHOLD).astype(int)
print()
print(f'Target distribution (score >= {TARGET_THRESHOLD}):')
vc = df['target'].value_counts()
for k, v in vc.items():
    print(f'  {k}: {v} rows ({v/len(df)*100:.1f}%)')


## Section 2 — Feature Leakage Audit

In [ ]:
direct_leakage_features = ['overall_opportunity_score', 'opportunity_tier']

formula_component_features = [
    'overall_demand_gap_score', 'expansion_readiness_score',
    'pricing_power_score', 'confidence_score',
    'overall_risk_score', 'markdown_safety_score'
]

secondary_score_features = ['adjusted_price_gap_pct', 'price_gap_confidence_weight']

raw_proxy_features = [
    'price_gap_pct', 'competitor_product_count', 'kroger_product_count',
    'google_trends_level_score', 'google_trends_momentum_score',
    'demand_velocity_score', 'ppi_3mo_trend', 'cpi_3mo_trend'
]

def keep(lst): return [f for f in lst if f in df.columns]
direct_leakage_features    = keep(direct_leakage_features)
formula_component_features = keep(formula_component_features)
secondary_score_features   = keep(secondary_score_features)
raw_proxy_features         = keep(raw_proxy_features)

print(f'{"Feature":<35} {"Leakage Level":<35} Reason')
print('-' * 110)
for f in direct_leakage_features:
    print(f'{f:<35} {"Direct target leakage":<35} IS the target or directly encodes it')
for f in formula_component_features:
    print(f'{f:<35} {"Formula component leakage":<35} Direct input to overall_opportunity_score formula')
for f in secondary_score_features:
    print(f'{f:<35} {"Derived / partial leakage":<35} Derived from scoring pipeline')
for f in raw_proxy_features:
    print(f'{f:<35} {"Allowed masked feature":<35} Upstream proxy; not a direct formula input')

print()
print('WARNING: Any model using direct or formula component features is an')
print('internal consistency check, not true predictive validation.')


## Section 3 — Evaluation Function

In [ ]:
def evaluate_model(df, feature_cols, label, model_type='rf'):
    sub = df.dropna(subset=['target']).copy()
    present = [c for c in feature_cols if c in sub.columns]
    X_all = sub[present].fillna(sub[present].median())
    y_all = sub['target'].values
    stores = sub['store_id'].values

    np.random.seed(RANDOM_STATE)
    unique_stores = np.unique(stores)
    test_stores   = np.random.choice(unique_stores, size=min(5, len(unique_stores)), replace=False)

    train_mask = ~np.isin(stores, test_stores)
    test_mask  =  np.isin(stores, test_stores)
    X_train, y_train = X_all[train_mask].values, y_all[train_mask]
    X_test,  y_test  = X_all[test_mask].values,  y_all[test_mask]

    if model_type == 'rf':
        model = RandomForestClassifier(n_estimators=200, max_depth=4,
                                       random_state=RANDOM_STATE, class_weight='balanced')
    else:
        model = LogisticRegression(max_iter=1000, class_weight='balanced',
                                   random_state=RANDOM_STATE)
    model.fit(X_train, y_train)
    y_pred  = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]

    auc = roc_auc_score(y_test, y_proba) if len(np.unique(y_test)) >= 2 else np.nan

    cm = confusion_matrix(y_test, y_pred)
    if cm.size == 4:
        tn, fp, fn, tp = cm.ravel()
    else:
        tn, fp, fn, tp = int(cm[0,0]), 0, 0, 0

    return {
        'label':     label,
        'features':  len(present),
        'AUC':       round(float(auc), 3) if not np.isnan(auc) else np.nan,
        'Precision': round(precision_score(y_test, y_pred, zero_division=0), 3),
        'Recall':    round(recall_score(y_test, y_pred, zero_division=0), 3),
        'F1':        round(f1_score(y_test, y_pred, zero_division=0), 3),
        'Accuracy':  round(accuracy_score(y_test, y_pred), 3),
        'TN': int(tn), 'FP': int(fp), 'FN': int(fn), 'TP': int(tp),
        '_model':    model,
        '_features': present,
        '_X_train':  X_train,
        '_y_train':  y_train,
    }

print('evaluate_model() defined.')


## Section 4 — Leaky vs Masked Comparison

In [ ]:
leaky_features       = formula_component_features + secondary_score_features + raw_proxy_features
masked_features      = raw_proxy_features
ultramasked_features = [
    'price_gap_pct', 'competitor_product_count', 'kroger_product_count',
    'google_trends_level_score', 'google_trends_momentum_score',
    'demand_velocity_score', 'ppi_3mo_trend', 'cpi_3mo_trend'
]

res_leaky     = evaluate_model(df, leaky_features,       'RF Leaky',        'rf')
res_masked_rf = evaluate_model(df, masked_features,      'RF Masked',       'rf')
res_masked_lr = evaluate_model(df, masked_features,      'LR Masked',       'lr')
res_ultra     = evaluate_model(df, ultramasked_features, 'RF Ultra-masked', 'rf')

all_results = [res_leaky, res_masked_rf, res_masked_lr, res_ultra]

def nan_str(v):
    return f'{v:.3f}' if not (isinstance(v, float) and v != v) else '  NaN'

hdr = f'{"Model":<18} {"Feats":>5} {"AUC":>6} {"Prec":>6} {"Rec":>6} {"F1":>6} {"Acc":>6} {"TN":>4} {"FP":>4} {"FN":>4} {"TP":>4}'
print(hdr)
print('-' * len(hdr))
for r in all_results:
    print(f'{r["label"]:<18} {r["features"]:>5} {nan_str(r["AUC"]):>6} '
          f'{r["Precision"]:>6.3f} {r["Recall"]:>6.3f} {r["F1"]:>6.3f} '
          f'{r["Accuracy"]:>6.3f} {r["TN"]:>4} {r["FP"]:>4} {r["FN"]:>4} {r["TP"]:>4}')

leaky_auc  = res_leaky['AUC']     if not (isinstance(res_leaky['AUC'], float) and res_leaky['AUC'] != res_leaky['AUC']) else 0.0
masked_auc = res_masked_rf['AUC'] if not (isinstance(res_masked_rf['AUC'], float) and res_masked_rf['AUC'] != res_masked_rf['AUC']) else 0.0
gap = leaky_auc - masked_auc
print(f'\nLeaky AUC: {leaky_auc:.3f}  |  Masked RF AUC: {masked_auc:.3f}  |  Gap: {gap:.3f}')


## Section 5 — K-Fold Stability Check

In [ ]:
def kfold_eval(df, feature_cols, label, n_splits=5):
    sub = df.dropna(subset=['target']).copy()
    present = [c for c in feature_cols if c in sub.columns]
    X = sub[present].fillna(sub[present].median()).values
    y = sub['target'].values
    aucs, precs, recs, f1s = [], [], [], []
    try:
        folds = list(StratifiedKFold(n_splits=n_splits, shuffle=True,
                                     random_state=RANDOM_STATE).split(X, y))
    except Exception:
        folds = list(KFold(n_splits=n_splits, shuffle=True,
                           random_state=RANDOM_STATE).split(X))
    for tr, te in folds:
        m = RandomForestClassifier(n_estimators=200, max_depth=4,
                                   random_state=RANDOM_STATE, class_weight='balanced')
        m.fit(X[tr], y[tr])
        yp  = m.predict(X[te])
        ypr = m.predict_proba(X[te])[:, 1]
        aucs.append(roc_auc_score(y[te], ypr) if len(np.unique(y[te])) >= 2 else np.nan)
        precs.append(precision_score(y[te], yp, zero_division=0))
        recs.append(recall_score(y[te], yp, zero_division=0))
        f1s.append(f1_score(y[te], yp, zero_division=0))
    a = np.array([x for x in aucs if not (isinstance(x, float) and x != x)])
    return {
        'label': label,
        'AUC mean': round(float(np.mean(a)),  3) if len(a) else np.nan,
        'AUC std':  round(float(np.std(a)),   3) if len(a) else np.nan,
        'Prec mean': round(float(np.mean(precs)), 3), 'Prec std': round(float(np.std(precs)), 3),
        'Recall mean': round(float(np.mean(recs)), 3), 'Recall std': round(float(np.std(recs)), 3),
        'F1 mean': round(float(np.mean(f1s)), 3), 'F1 std': round(float(np.std(f1s)), 3),
    }

kf_leaky  = kfold_eval(df, leaky_features,       'Leaky')
kf_masked = kfold_eval(df, masked_features,      'Masked')
kf_ultra  = kfold_eval(df, ultramasked_features, 'Ultra-masked')

print(f'{"Feature Set":<14} {"AUC mean":>9} {"AUC std":>8} {"Prec mean":>10} {"Prec std":>9} {"Rec mean":>9} {"Rec std":>8} {"F1 mean":>8} {"F1 std":>8}')
print('-' * 95)
for r in [kf_leaky, kf_masked, kf_ultra]:
    am = nan_str(r['AUC mean']); as_ = nan_str(r['AUC std'])
    print(f'{r["label"]:<14} {am:>9} {as_:>8} {r["Prec mean"]:>10.3f} {r["Prec std"]:>9.3f} '
          f'{r["Recall mean"]:>9.3f} {r["Recall std"]:>8.3f} {r["F1 mean"]:>8.3f} {r["F1 std"]:>8.3f}')

gap_kf = 0.0
if not (isinstance(kf_leaky['AUC mean'], float) and kf_leaky['AUC mean'] != kf_leaky['AUC mean']):
    if not (isinstance(kf_masked['AUC mean'], float) and kf_masked['AUC mean'] != kf_masked['AUC mean']):
        gap_kf = kf_leaky['AUC mean'] - kf_masked['AUC mean']
print(f'\nK-fold AUC gap (Leaky - Masked): {gap_kf:.3f}')
if gap_kf >= 0.10:
    print('  -> Leakage confirmed: leaky model significantly outperforms masked model')
elif kf_masked['AUC mean'] >= 0.80:
    print('  -> Partial leakage: raw/proxy features reconstruct the score well')
else:
    print('  -> Gap small or both low — dataset may be too small/flat for strong ML signal')


## Section 6 — Permutation Target Sanity Check

In [ ]:
sub = df.dropna(subset=['target']).copy()
present = [c for c in masked_features if c in sub.columns]
X_all = sub[present].fillna(sub[present].median()).values
y_real = sub['target'].values

np.random.seed(RANDOM_STATE)
unique_stores = np.unique(sub['store_id'].values)
test_stores   = np.random.choice(unique_stores, size=min(5, len(unique_stores)), replace=False)
train_mask = ~np.isin(sub['store_id'].values, test_stores)
test_mask  =  np.isin(sub['store_id'].values, test_stores)

y_shuffled = sub['target'].sample(frac=1, random_state=99).reset_index(drop=True).values
X_train, X_test = X_all[train_mask], X_all[test_mask]
y_tr_shuf, y_te_real = y_shuffled[train_mask], y_real[test_mask]

m_shuf = RandomForestClassifier(n_estimators=200, max_depth=4,
                                 random_state=RANDOM_STATE, class_weight='balanced')
m_shuf.fit(X_train, y_tr_shuf)
y_pred_shuf  = m_shuf.predict(X_test)
y_proba_shuf = m_shuf.predict_proba(X_test)[:, 1]

shuf_auc  = roc_auc_score(y_te_real, y_proba_shuf) if len(np.unique(y_te_real)) >= 2 else np.nan
shuf_prec = precision_score(y_te_real, y_pred_shuf, zero_division=0)
shuf_f1   = f1_score(y_te_real, y_pred_shuf, zero_division=0)

print('Shuffled-target sanity check (masked features, real test labels):')
print(f'  AUC:       {nan_str(shuf_auc)}')
print(f'  Precision: {shuf_prec:.3f}')
print(f'  F1:        {shuf_f1:.3f}')
print()
shuf_auc_val = shuf_auc if not (isinstance(shuf_auc, float) and shuf_auc != shuf_auc) else 0.0
if shuf_auc_val > 0.70:
    print('RED FLAG: model may still be leaking through index/order or duplicated rows.')
else:
    print('OK: Shuffled-target AUC is low — no index or duplication leakage detected.')


## Section 7 — Feature Importance Comparison

In [ ]:
def plot_imp(model, feats, title, path, top=10):
    imp = pd.Series(model.feature_importances_, index=feats).sort_values(ascending=False).head(top)
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.barh(imp.index[::-1], imp.values[::-1], color='#4C72B0')
    ax.set_xlabel('Importance (mean decrease in impurity)')
    ax.set_title(title)
    plt.tight_layout()
    plt.savefig(path, dpi=120)
    plt.show()
    print(f'Saved: {path}')
    return imp

print('=== Leaky model — top 10 features ===')
leaky_imp = plot_imp(res_leaky['_model'], res_leaky['_features'],
    'Leaky Model Feature Importance (formula components + raw proxies)',
    'outputs/leaky_feature_importance.png')
print(leaky_imp.round(4).to_string())

print('\n=== Masked model — top 10 features ===')
masked_imp = plot_imp(res_masked_rf['_model'], res_masked_rf['_features'],
    'Masked Model Feature Importance (raw/upstream proxy features only)',
    'outputs/masked_feature_importance.png')
print(masked_imp.round(4).to_string())

leaky_top  = leaky_imp.index[0]
masked_top = masked_imp.index[0]
print()
if leaky_top in formula_component_features:
    print(f'Leaky top feature "{leaky_top}" is a formula component — leakage confirmed.')
else:
    print(f'Leaky top feature "{leaky_top}" is a raw/proxy signal.')
if masked_top in raw_proxy_features:
    print(f'Masked top feature "{masked_top}" is raw/proxy — expected.')


## Section 8 — Final Verdict

In [ ]:
print('=' * 70)
print('FINAL VERDICT')
print('=' * 70)
print(f'Leaky model AUC:  {leaky_auc:.3f}')
print(f'Masked model AUC: {masked_auc:.3f}')
print(f'AUC gap:          {gap:.3f}')
print()

if gap >= 0.10:
    verdict = ('Leakage confirmed: previous model was learning the score formula. '
               'The leaky model significantly outperforms the masked model because '
               'formula component features directly encode the target.')
elif leaky_auc >= 0.90 and masked_auc >= 0.80:
    verdict = ('Partial leakage likely, but raw/proxy features also reconstruct '
               'the score well. Upstream demand and pricing signals contain genuine '
               'predictive signal, but the model benefits from formula component overlap.')
else:
    verdict = ('No strong leakage gap detected, but target is still rule-derived. '
               'Both models show similar performance at this dataset size.')

always_add = ('This is not real business outcome validation because the target is '
              'derived from TrendShelf scoring logic, not sales, margin, or unit-volume outcomes.')

print('VERDICT:', verdict)
print()
print('NOTE:', always_add)
print('=' * 70)

VERDICT_TEXT = verdict
ALWAYS_ADD   = always_add


## Section 9 — Write Markdown Report

In [ ]:
import os

def fv(v):
    return nan_str(v)

# Build table rows
leak_rows = []
for f in direct_leakage_features:
    leak_rows.append('| ' + f + ' | Direct target leakage | IS the target or directly encodes it |')
for f in formula_component_features:
    leak_rows.append('| ' + f + ' | Formula component leakage | Direct input to opportunity_score formula |')
for f in secondary_score_features:
    leak_rows.append('| ' + f + ' | Derived / partial leakage | Derived from scoring pipeline |')
for f in raw_proxy_features:
    leak_rows.append('| ' + f + ' | Allowed masked feature | Upstream proxy; not a direct formula input |')

comp_rows = []
for r in all_results:
    comp_rows.append('| ' + r['label'] + ' | ' + str(r['features']) + ' | ' + fv(r['AUC'])
                     + ' | ' + str(r['Precision']) + ' | ' + str(r['Recall'])
                     + ' | ' + str(r['F1']) + ' | ' + str(r['Accuracy'])
                     + ' | ' + str(r['TN']) + ' | ' + str(r['FP'])
                     + ' | ' + str(r['FN']) + ' | ' + str(r['TP']) + ' |')

kf_rows = []
for r in [kf_leaky, kf_masked, kf_ultra]:
    kf_rows.append('| ' + r['label'] + ' | ' + fv(r['AUC mean']) + ' | ' + fv(r['AUC std'])
                   + ' | ' + str(r['Prec mean']) + ' | ' + str(r['Prec std'])
                   + ' | ' + str(r['Recall mean']) + ' | ' + str(r['Recall std'])
                   + ' | ' + str(r['F1 mean']) + ' | ' + str(r['F1 std']) + ' |')

report_lines = [
    '# TrendShelf Scoring Leakage Audit', '',
    '## Purpose',
    'Check whether the ML validation model was learning the rule-based opportunity score formula.', '',
    '## Target', '`target = overall_opportunity_score >= 65`', '',
    '## Why leakage is possible',
    '`overall_opportunity_score` is created from component scores such as demand gap, expansion readiness,',
    'pricing power, confidence, risk safety, and markdown safety. If these same features are used to',
    'train a model, the model may simply reconstruct the formula.', '',
    '## Tests Run',
    '1. Feature leakage audit', '2. Leaky vs masked model comparison',
    '3. K-fold stability check', '4. Shuffled-target sanity check',
    '5. Feature importance review', '',
    '## Results', '',
    '### Feature Leakage Audit',
    '| Feature | Leakage Level | Reason |', '|---------|---------------|--------|',
] + leak_rows + ['',
    '### Leaky vs Masked Comparison',
    '| Model | Features | AUC | Precision | Recall | F1 | Accuracy | TN | FP | FN | TP |',
    '|-------|----------|-----|-----------|--------|----|----------|----|----|----|----|',
] + comp_rows + ['',
    '### K-Fold Stability (5-fold, RandomForest)',
    '| Feature Set | AUC mean | AUC std | Prec mean | Prec std | Recall mean | Recall std | F1 mean | F1 std |',
    '|-------------|----------|---------|-----------|----------|-------------|------------|---------|--------|',
] + kf_rows + ['',
    '### Shuffled-Target Sanity Check',
    '| Check | Value |', '|-------|-------|',
    '| Shuffled-target AUC | ' + fv(shuf_auc) + ' |',
    '| Shuffled-target Precision | ' + str(round(shuf_prec, 3)) + ' |',
    '| Shuffled-target F1 | ' + str(round(shuf_f1, 3)) + ' |', '',
    '### Top Feature Importances',
    '**Leaky model top feature:** `' + leaky_top + '` (importance=' + str(round(leaky_imp.iloc[0], 4)) + ')',
    '**Masked model top feature:** `' + masked_top + '` (importance=' + str(round(masked_imp.iloc[0], 4)) + ')', '',
    '## Verdict', VERDICT_TEXT, '',
    '## Interpretation',
    'If the leaky model performs much better than the masked model, the original model should be treated',
    'only as internal consistency validation, not a true predictive backtest.', '',
    '## Correct README wording',
    '> "Initial ML validation produced very high performance because the model used component scores that',
    '> also define the opportunity target. I treated this as an internal consistency check and then ran',
    '> a masked leakage audit using only upstream proxy features. True outcome validation will require',
    '> future sales, margin, unit-volume, or multi-month action outcome labels."', '',
    '## Limitations',
    '- 200 rows', '- 1 month of scoring data', '- Target is rule-derived',
    '- No sales/profit/unit-volume label', '- Store-level holdout only',
    '- True validation needs 3-6+ months of history',
]

os.makedirs('docs', exist_ok=True)
report_text = '\n'.join(report_lines)
with open('docs/scoring_leakage_audit.md', 'w', encoding='utf-8') as f:
    f.write(report_text)
print('Saved: docs/scoring_leakage_audit.md')
print()
print(report_text[:600])
